In [7]:
import os
import torch
import random
import numpy as np
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# For training and logging
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision

In [8]:
# ------------------
# Dataset Definition
# ------------------
class MyRatDataset(Dataset):
    """
    A dataset that reads images and YOLO-format annotations from a folder.
    Expected structure:
        root/
          images/
            img1.jpg
            img2.jpg
            ...
          labels/
            img1.txt
            img2.txt
            ...
    Each label file should contain lines (YOLO format):
        class_id  x_center_norm  y_center_norm  width_norm  height_norm
    Negative samples can be created on the fly with a given probability.
    This updated version prints a warning and ignores any image whose label file contains negative values.
    """
    def __init__(self, root, transforms=None, negative_sample_ratio=0.0):
        self.root = root
        self.transforms = transforms
        self.negative_sample_ratio = negative_sample_ratio
        
        self.img_dir = os.path.join(root, "images")
        self.lbl_dir = os.path.join(root, "labels")
        
        # Collect image file names
        self.imgs = [f for f in os.listdir(self.img_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.imgs.sort()

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(self.lbl_dir, label_name)
        
        boxes = []
        labels = []
        corrupt = False
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    try:
                        class_id = int(parts[0]) + 1  # shift if needed; reserve 0 for background
                        x_center_norm = float(parts[1])
                        y_center_norm = float(parts[2])
                        width_norm    = float(parts[3])
                        height_norm   = float(parts[4])
                    except ValueError:
                        continue

                    if x_center_norm < 0 or y_center_norm < 0 or width_norm < 0 or height_norm < 0:
                        # Print a warning and treat image as negative sample
                        print(f"WARNING {img_path}: ignoring corrupt image/label: negative label values "
                              f"[{x_center_norm:.5f}, {y_center_norm:.5f}, {width_norm:.5f}, {height_norm:.5f}]")
                        corrupt = True
                        break

                    x_center = x_center_norm * w
                    y_center = y_center_norm * h
                    box_width = width_norm * w
                    box_height = height_norm * h
                    x_min = x_center - box_width / 2
                    y_min = y_center - box_height / 2
                    x_max = x_center + box_width / 2
                    y_max = y_center + box_height / 2
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(class_id)
        
        if corrupt:
            boxes = []
            labels = []
        
        if len(boxes) == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Optionally generate a negative sample
        if boxes.shape[0] > 0 and random.random() < self.negative_sample_ratio:
            img = self.generate_negative_sample(img, boxes)
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        
        image_id = torch.tensor([idx])
        if boxes.size(0) > 0:
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        else:
            area = torch.empty((0,), dtype=torch.float32)
        iscrowd = torch.zeros((labels.shape[0],), dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }
        
        if self.transforms:
            img = self.transforms(img)
        
        return img, target

    def generate_negative_sample(self, img, boxes):
        img_np = np.array(img)
        for box in boxes:
            x_min, y_min, x_max, y_max = box.int().tolist()
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_np.shape[1], x_max)
            y_max = min(img_np.shape[0], y_max)
            if x_max > x_min and y_max > y_min:
                noise = np.random.randint(0, 256, (y_max - y_min, x_max - x_min, 3), dtype=np.uint8)
                img_np[y_min:y_max, x_min:x_max, :] = noise
        return Image.fromarray(img_np)


In [9]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained = True)

In [10]:
num_classes = 2
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [ ]:
# Define transforms (resize to 300x300 to match SSD300 input)
transforms = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Create training and validation datasets
train_dataset = MyRatDataset(root="new_dataset/train", transforms=transforms)
val_dataset   = MyRatDataset(root="new_dataset/valid", transforms=transforms)

# Create dataloaders with an appropriate collate function
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=lambda batch: tuple(zip(*batch)),
    # pin_memory = True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda batch: tuple(zip(*batch)),
    # pin_memory = True if torch.cuda.is_available() else False
)

In [13]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

# Clear cached memory before moving the model to the device.
torch.cuda.empty_cache()

# Set the device to GPU if available.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Try to move the model to the selected device.
try:
    model.to(device)
except RuntimeError as e:
    if 'out of memory' in str(e):
        print("CUDA out-of-memory error encountered. Falling back to CPU.")
        # Optionally, you might want to release more memory:
        torch.cuda.empty_cache()
        device = torch.device('cpu')
        model.to(device)
    else:
        raise e


# Set up optimizer and scheduler
optimizer = optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)
num_epochs = 10
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
scaler = GradScaler()

# For logging
writer = SummaryWriter(log_dir="runs")

# Lists for plotting
train_losses = []
val_losses = []
train_cls_losses = []
val_cls_losses = []
train_bbox_losses = []
val_bbox_losses = []

# Helper: Freeze BatchNorm layers if desired.
def freeze_bn(module):
    if isinstance(module, torch.nn.BatchNorm2d):
        module.eval()

best_val_loss = float('inf')
best_epoch = -1

try:
    for epoch in range(num_epochs):
        model.train()
        model.apply(freeze_bn)
        
        epoch_train_loss = 0.0
        epoch_train_cls_loss = 0.0
        epoch_train_bbox_loss = 0.0
        
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Optionally filter o5ut samples with empty boxes (if needed)
            filtered_images = []
            filtered_targets = []
            for img, tgt in zip(images, targets):
                if tgt["boxes"].numel() > 0:
                    filtered_images.append(img)
                    filtered_targets.append(tgt)
            
            optimizer.zero_grad()
            with autocast():
                if len(filtered_images) == 0:
                    total_loss = torch.tensor(0., device=device, requires_grad=True)
                else:
                    loss_dict = model(filtered_images, filtered_targets)
                    cls_loss = loss_dict["loss_classifier"]
                    bbox_loss = loss_dict["loss_box_reg"]
                    rpn_cls_loss = loss_dict["loss_objectness"]
                    rpn_bbox_loss = loss_dict["loss_rpn_box_reg"]
                    total_loss = cls_loss + bbox_loss
            
            if total_loss.item() != 0:
                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            
            epoch_train_loss += total_loss.item()
            if len(filtered_images) > 0:
                epoch_train_cls_loss += cls_loss.item()
                epoch_train_bbox_loss += bbox_loss.item()
        
        epoch_train_loss /= len(train_loader)
        epoch_train_cls_loss /= len(train_loader)
        epoch_train_bbox_loss /= len(train_loader)
        train_losses.append(epoch_train_loss)
        train_cls_losses.append(epoch_train_cls_loss)
        train_bbox_losses.append(epoch_train_bbox_loss)
        
        writer.add_scalar("Loss/Train/Total", epoch_train_loss, epoch)
        writer.add_scalar("Loss/Train/Class", epoch_train_cls_loss, epoch)
        writer.add_scalar("Loss/Train/BBox", epoch_train_bbox_loss, epoch)
        print(f"Epoch {epoch+1}/{num_epochs}: Train Total Loss: {epoch_train_loss:.4f}, "
              f"Train cls Loss: {epoch_train_cls_loss:.4f}, Train bbox Loss: {epoch_train_bbox_loss:.4f}")
        
        # Validation loop
        model.train()  # SSD returns losses only in train mode.
        model.apply(freeze_bn)
        
        epoch_val_loss = 0.0
        epoch_val_cls_loss = 0.0
        epoch_val_bbox_loss = 0.0
        
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                
                filtered_images = []
                filtered_targets = []
                for img, tgt in zip(images, targets):
                    if tgt["boxes"].numel() > 0:
                        filtered_images.append(img)
                        filtered_targets.append(tgt)
                
                if len(filtered_images) == 0:
                    total_loss = torch.tensor(0., device=device)
                else:
                    loss_dict = model(filtered_images, filtered_targets)
                    cls_loss = loss_dict["loss_classifier"]
                    bbox_loss = loss_dict["loss_box_reg"]
                    rpn_cls_loss = loss_dict["loss_objectness"]
                    rpn_bbox_loss = loss_dict["loss_rpn_box_reg"]
                    total_loss = cls_loss + bbox_loss
                
                epoch_val_loss += total_loss.item()
                if len(filtered_images) > 0:
                    epoch_val_cls_loss += cls_loss.item()
                    epoch_val_bbox_loss += bbox_loss.item()
        
        epoch_val_loss /= len(val_loader)
        epoch_val_cls_loss /= len(val_loader)
        epoch_val_bbox_loss /= len(val_loader)
        val_losses.append(epoch_val_loss)
        val_cls_losses.append(epoch_val_cls_loss)
        val_bbox_losses.append(epoch_val_bbox_loss)
        
        writer.add_scalar("Loss/Val/Total", epoch_val_loss, epoch)
        writer.add_scalar("Loss/Val/Class", epoch_val_cls_loss, epoch)
        writer.add_scalar("Loss/Val/BBox", epoch_val_bbox_loss, epoch)
        print(f"Epoch {epoch+1}/{num_epochs}: Val Total Loss: {epoch_val_loss:.4f}, "
              f"Val cls Loss: {epoch_val_cls_loss:.4f}, Val bbox Loss: {epoch_val_bbox_loss:.4f}")
        
        optimizer.step()
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), "Best_Model_FastRCNN.pth")
            print(f"Best model saved at epoch {best_epoch} with val loss {best_val_loss:.4f}")
    
except KeyboardInterrupt:
    print("Training interrupted. Saving best model so far...")

torch.save(model.state_dict(), "Final_Model_FastRCNN.pth")
print(f"Final model saved to Final_Model.pth")
print(f"Best model was from epoch {best_epoch} with val loss {best_val_loss:.4f}")


C:\Users\mzarrar\AppData\Local\Temp\ipykernel_13628\1565004495.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/10:   0%|          | 0/136 [00:00<?, ?it/s]C:\Users\mzarrar\AppData\Local\Temp\ipykernel_13628\1565004495.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/10:  41%|████      | 56/136 [29:17<1:04:57, 48.72s/it]

: 